# Line Crossing Event Detection

Validate the `LineCrossingDetector` end-to-end: draw a virtual trip line on a real video frame, preview it across the clip, then run the detector on the full video to confirm a crossing event fires with the correct `track_id`.

**What this notebook validates:**
- Interactive line definition via OpenCV mouse callback
- Visual sanity check that the line sits at the intended position
- `LineCrossingDetector.check()` produces one event per track per crossing direction

**Test video:** `GX012761.MP4` (a person walks across the scene once)

## 1. Draw the line

Open the first frame of the video and click two points with the left mouse button. A red dot marks each click, a green line is drawn between the two points, and the pixel coordinates are printed for reuse in the detector cell below.

Press any key after clicking to close the window.

In [5]:
import cv2

cap = cv2.VideoCapture("C:/Users/gmission/Desktop/video-test/GX012761.MP4")
ret, frame = cap.read()
cap.release()

points = []

def click(event, x, y, flags, param):
    if event == cv2.EVENT_LBUTTONDOWN:
        points.append((x, y))
        cv2.circle(frame, (x, y), 5, (0, 0, 255), -1)
        if len(points) == 2:
            cv2.line(frame, points[0], points[1], (0, 255, 0), 2)
        cv2.imshow("Draw Line - Click 2 points", frame)

cv2.imshow("Draw Line - Click 2 points", frame)
cv2.setMouseCallback("Draw Line - Click 2 points", click)
cv2.waitKey(0)
cv2.destroyAllWindows()

print(f"Line: start={points[0]}, end={points[1]}")

Line: start=(514, 741), end=(921, 794)


## 2. Preview the line on video

Play back the full clip with the line overlaid. Useful to confirm the line covers the expected crossing path and does not drift because of camera motion. Press `q` to quit early.

In [6]:
import cv2

cap = cv2.VideoCapture("C:/Users/gmission/Desktop/video-test/GX012761.MP4")

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    cv2.line(frame, (514, 741), (921, 794), (0, 255, 0), 2)
    cv2.imshow("Line Position", frame)
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

## 3. Run the detector

Feed each frame through `YOLO.track()` (person class only) and pass the results to `LineCrossingDetector.check()`. The detector maintains per-track state and fires an event the moment a track's center transitions from one side of the line to the other.

- `classes=[0]` restricts detections to COCO class `person`.
- `conf=0.5` filters low-confidence boxes (tracker noise).
- Loop caps at 1000 frames to keep the cell fast on long clips.

In [7]:
import sys
sys.path.append("../ai/src")
import cv2
from ultralytics import YOLO
from events.line_crossing import LineCrossingDetector

model = YOLO("yolo11n.pt")
# Replace with your line coordinates
detector = LineCrossingDetector(line_start=(514, 741), line_end=(921, 794))

cap = cv2.VideoCapture("C:/Users/gmission/Desktop/video-test/GX012761.MP4")

frame_count = 0
while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    results = model.track(frame, persist=True, conf=0.5, classes=[0], verbose=False)
    events = detector.check(results)

    for e in events:
        print(f"Frame {frame_count}: Line Crossing ID {e['track_id']}")

    frame_count += 1
    if frame_count > 1000:
        break

cap.release()
print("Done")

Frame 780: Line Crossing ID 3
Done


## 4. Results

| Event        | Frame | Track ID |
|--------------|-------|----------|
| Line crossed | 780   | 3        |

- Exactly **one** event fired across ~1000 frames, matching the ground truth (one person walks across once).
- No duplicate events on subsequent frames → per-track side-state is held correctly.
- `LineCrossingDetector` is wired into the pipeline and ready for production use.